In [1]:
import os
import pandas as pd
import yaml
import pickle

from utils.training_utils import find_specific_variables

import xgboost as xgb

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [2]:
features = yaml.safe_load(open(os.path.join('..', 'src', 'config', 'feature_config.yaml'), 'r'))

# Modelo para classificação de um produto em promoção

In [3]:
df = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))

print(df.shape)
df.head()

(32940, 14)


,contact,default,education,job,marital,month,poutcome,quarter,age,campaign,contacts_tendency,pdays,previous,y
0,0.0,0.0,6.0,0.0,2.0,9.0,1.0,2.0,31.0,3.0,0.0,999.0,0.0,0
1,1.0,0.0,3.0,3.0,1.0,6.0,1.0,1.0,39.0,2.0,0.0,999.0,0.0,0
2,0.0,0.0,5.0,2.0,1.0,3.0,1.0,2.0,34.0,4.0,0.0,999.0,0.0,0
3,1.0,1.0,2.0,9.0,1.0,6.0,1.0,1.0,36.0,9.0,0.0,999.0,0.0,0
4,0.0,1.0,7.0,8.0,2.0,1.0,1.0,2.0,25.0,1.0,0.0,999.0,0.0,0


In [4]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

df_hyperparams = pickle.load(
    open(os.path.join('..', 'models', 'df_metrics_results_tunning.pkl'), 'rb')
)

In [5]:

feature_target = find_specific_variables(features, 'target', specific_value=True)

In [6]:
df_treino, df_valid = train_test_split(df, test_size=0.2, random_state=96)

In [7]:
print(f'Shape Treino: {df_treino.shape}')
print(f'Shape Valid: {df_valid.shape}')

Shape Treino: (26352, 14)
Shape Valid: (6588, 14)


In [8]:
print(f'% Treino: {df_treino[feature_target[0]].mean()}')
print(f'% Valid: {df_valid[feature_target[0]].mean()}')

% Treino: 0.11202185792349727
% Valid: 0.11262902246508803


In [9]:
df_hyperparams[df_hyperparams.value == max(df_hyperparams.value)].T

,37
number,37
value,0.779447
datetime_start,2025-03-30 12:39:33.686332
datetime_complete,2025-03-30 12:39:34.007332
duration,0 days 00:00:00.321000
params_colsample_bytree,0.830086
params_gamma,0.083646
params_learning_rate,0.039425
params_max_depth,6
params_min_child_weight,45.060789


In [10]:
best_row = df_hyperparams.loc[df_hyperparams['value'].idxmax()]
best_params = best_row.filter(like='params_')
hyper_params = {col.replace('params_', ''): best_params[col] for col in best_params.index}


hyper_params.update({
    'eval_metric': 'auc',
})

In [11]:
hyper_params

{'colsample_bytree': 0.8300864657880125,
 'gamma': 0.08364560379320828,
 'learning_rate': 0.03942452234009175,
 'max_depth': 6,
 'min_child_weight': 45.06078916507349,
 'scale_pos_weight': 29.745685425601856,
 'subsample': 0.8052744378444359,
 'eval_metric': 'auc'}

In [12]:
model = xgb.XGBClassifier(
    **hyper_params,
    random_state=12,
    n_jobs=-1,
    early_stopping_rounds=4
)

model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8300864657880125, device=None,
              early_stopping_rounds=4, enable_categorical=False,
              eval_metric='auc', feature_types=None, gamma=0.08364560379320828,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03942452234009175,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=45.06078916507349, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=-1, num_parallel_tree=None, random_state=12, ...)

In [13]:
model.fit(
    df_treino[seletor.features].values,
    df_treino[feature_target].values,
    eval_set=[(df_valid[seletor.features].values, df_valid[feature_target].values)],
    verbose=True
)

[0]	validation_0-auc:0.75335
[1]	validation_0-auc:0.75861
[2]	validation_0-auc:0.76571
[3]	validation_0-auc:0.76949
[4]	validation_0-auc:0.76967
[5]	validation_0-auc:0.77395
[6]	validation_0-auc:0.77560
[7]	validation_0-auc:0.77628
[8]	validation_0-auc:0.77871
[9]	validation_0-auc:0.77981
[10]	validation_0-auc:0.78045
[11]	validation_0-auc:0.78151
[12]	validation_0-auc:0.78034
[13]	validation_0-auc:0.77971
[14]	validation_0-auc:0.77924
[15]	validation_0-auc:0.77976


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8300864657880125, device=None,
              early_stopping_rounds=4, enable_categorical=False,
              eval_metric='auc', feature_types=None, gamma=0.08364560379320828,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03942452234009175,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=45.06078916507349, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=-1, num_parallel_tree=None, random_state=12, ...)

In [14]:
pickle.dump(
    model, 
    open(os.path.join('..', 'models', 'predictors', 'model.pkl'), 'wb')
)